# RAG Experiment Notebook
This notebook contains a standalone implementation of Retrieval-Augmented Generation (RAG). You can modify the retrieval logic, the prompt, and the language model here without affecting the production services.

In [3]:
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Ensure the project root is in the path so we can import our modules
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from api.dependencies.containers import get_dense_service, get_db_store

### 1. Initialize Components
We load the retriever, the document store, and initialize the LLM directly in the notebook.

In [4]:
# 1. Initialize Retriever (Dense Retriever)
retriever = get_dense_service()

# 2. Initialize Document Store (SQLite Database)
doc_store = get_db_store()

# 3. Initialize Language Model (Hugging Face)
model_name = "google/flan-t5-base"
print(f"Loading LLM: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Components initialized successfully!")

Loading LLM: google/flan-t5-base...


KeyboardInterrupt: 

### 2. Define the RAG Logic
This function mimics the `RagUseCase` but is fully exposed so you can experiment with the prompt formatting and generation parameters.

In [ ]:
def experiment_rag(query: str, top_k: int = 3, max_new_tokens: int = 100):
    # A. Retrieve top-k documents
    print(f"Retrieving top {top_k} documents for query: '{query}'")
    retrieved_docs = retriever.retrieve(query, top_k)
    
    # B. Fetch full context from the database
    context_texts = []
    for d in retrieved_docs:
        text = doc_store.get_document_text(d.doc_id)
        if text:
            context_texts.append(text[:500]) # Taking first 500 chars per doc
            
    context = "\n---\n".join(context_texts)
    
    # C. Build the Prompt (Modify this to test different prompt engineering strategies)
    prompt = (
        "Answer the question using ONLY the provided context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{query}\n\n"
        "Answer:\n"
    )
    print("\n--- PROMPT ---")
    print(prompt)
    print("--------------\n")
    
    # D. Generate Answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer, retrieved_docs

### 3. Test the RAG Pipeline
Run a test query through our experimental RAG implementation.

In [ ]:
query = "Should we ban abortion?" # Example dataset topic
answer, docs = experiment_rag(query, top_k=3, max_new_tokens=50)

print("\n========== FINAL RAG ANSWER ==========")
print(answer)
print("======================================\n")
print("Retrieved Document IDs:", [d.doc_id for d in docs])